# Chapter 6: User Story Generation

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an **Agile Story Generator** that takes high-level features or epics and breaks them down into well-structured user stories with acceptance criteria, story points, and sprint assignments.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Familiarity with Agile/Scrum concepts


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Epic to User Stories Decomposition

Take a high-level epic and break it down into appropriately sized user stories.


In [ ]:
# Define an epic
epic = {
    'id': 'EPIC-101',
    'title': 'User Authentication & Authorization',
    'description': 'Implement a complete authentication system with login, registration, password management, and role-based access control for the e-commerce platform.',
    'target_users': ['Customers', 'Admin Users', 'Store Managers'],
    'business_value': 'Security compliance and personalized user experience'
}

# Generate user stories from epic
story_prompt = f"""Break down this epic into user stories following the INVEST criteria 
(Independent, Negotiable, Valuable, Estimable, Small, Testable).

Epic: {json.dumps(epic)}

For each user story, provide:
- id: STORY-1xx format
- title: short descriptive title
- as_a: user role
- i_want: desired action
- so_that: business value
- acceptance_criteria: list of Given/When/Then scenarios (at least 3)
- story_points: fibonacci (1, 2, 3, 5, 8)
- priority: MoSCoW (Must, Should, Could, Won't)
- dependencies: list of other story IDs this depends on

Generate 6-8 user stories. Return as a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': story_prompt}],
    temperature=0.3
)
stories = json.loads(response.choices[0].message.content)
print(f'Generated {len(stories)} user stories from epic "{epic["title"]}"')
for s in stories:
    print(f"\n{s['id']}: {s['title']}")
    print(f"  As a {s['as_a']}, I want {s['i_want'][:60]}...")
    print(f"  Points: {s['story_points']} | Priority: {s['priority']} | AC: {len(s['acceptance_criteria'])}")

## 2. Acceptance Criteria Refinement

Take a user story and generate comprehensive, edge-case-aware acceptance criteria.


In [ ]:
# Pick the first story and refine its acceptance criteria
story = stories[0]

refine_prompt = f"""As a senior BA, refine and expand the acceptance criteria for this user story.
Consider: happy paths, error paths, edge cases, security, performance, and accessibility.

User Story: {json.dumps(story)}

For each acceptance criterion:
- Use Given/When/Then format
- Tag as: happy_path, error_path, edge_case, security, performance, or accessibility
- Add a testability_note explaining how a QA engineer would verify it

Return a JSON object with:
- story_id: the story ID
- refined_criteria: array of {{given, when, then, tag, testability_note}}

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': refine_prompt}],
    temperature=0.3
)
refined = json.loads(response.choices[0].message.content)
print(f"Refined acceptance criteria for {refined['story_id']}:\n")
for i, ac in enumerate(refined['refined_criteria'], 1):
    print(f"{i}. [{ac['tag'].upper()}]")
    print(f"   Given: {ac['given']}")
    print(f"   When: {ac['when']}")
    print(f"   Then: {ac['then']}")
    print()

In [ ]:
# Sprint planning: assign stories to sprints based on dependencies and capacity
sprint_prompt = f"""Given these user stories, create a sprint plan.

Stories: {json.dumps(stories)}

Constraints:
- Sprint duration: 2 weeks
- Team velocity: 20 story points per sprint
- Must respect dependency order
- Must-have stories go first

Return a sprint plan as a JSON object with:
- sprints: array of {{sprint_number, goal, stories (list of IDs), total_points, capacity_used_pct}}
- risks: any concerns about the plan

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': sprint_prompt}],
    temperature=0.3
)
plan = json.loads(response.choices[0].message.content)
for sprint in plan['sprints']:
    print(f"\nSprint {sprint['sprint_number']}: {sprint['goal']}")
    print(f"  Stories: {', '.join(sprint['stories'])}")
    print(f"  Points: {sprint['total_points']} | Capacity: {sprint['capacity_used_pct']}%")

## Exercises
1. Take a real epic from your project and generate user stories. Compare with what your team created manually.
2. Add a story-splitting function that breaks stories > 8 points into smaller ones.
3. Build a story quality checker that validates INVEST criteria.
4. Create a function that generates a Definition of Done checklist for each story.
